[![nbviewer](https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg)](https://nbviewer.org/github/maciejskorski/ml_examples/blob/master/src/RFT_demo.ipynb?flush_cache=true)

# Summary

This experiment uses the reinforcement fine-tunning (RFT) service of OpenAI to fine-tune one of their modes for better emotion recognition.

Emotion recognition is a subjective multilabel classification problem, on which LLMs suffer from overly sparse predictions.

In [2]:
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv() # load .env file, including OPENAI tokens
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,5"  # Hide GPU 4

In [4]:
from datasets import load_dataset

ds = load_dataset('SemEvalWorkshop/sem_eval_2018_task_1', 'subtask5.english')
print(ds)

DatasetDict({
    train: Dataset({
        features: ['ID', 'Tweet', 'anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust'],
        num_rows: 6838
    })
    test: Dataset({
        features: ['ID', 'Tweet', 'anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust'],
        num_rows: 3259
    })
    validation: Dataset({
        features: ['ID', 'Tweet', 'anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust'],
        num_rows: 886
    })
})


In [5]:
emotion_labels = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']

def extract_emotions(example):
    active_indices = tuple(col for i, col in enumerate(emotion_labels) if example[col] == 1)
    return {"labels": active_indices}

ds = ds.map(extract_emotions)
ds['validation']['labels'][:10]

[['anger', 'disgust'],
 ['joy', 'love', 'optimism'],
 ['anger', 'disgust'],
 ['joy', 'optimism'],
 ['fear'],
 ['anger', 'disgust'],
 ['anger', 'disgust'],
 ['anger', 'disgust'],
 ['anger', 'disgust'],
 ['disgust', 'pessimism', 'sadness']]

# Reinforcement Fine-Tuning

We need to prepare the dataset for RFT. 

The reward is `Intersection over Union` as computed by the model grader (implementing the algebraic variant was difficult).

The model was fine-tuned on the validation part and evaluated on the test part.


In [5]:
import json 

def get_rft_example(example):
    emotion_labels = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 
                      'optimism', 'pessimism', 'sadness', 'surprise', 'trust']
    
    text = example['Tweet']
    user_msg = f"You are the emotion recognition system. Give a comma-separeted list of emotions, out of ('anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust') posibilities, that are present in the message: {text}"
    labels = tuple(col for col in emotion_labels if example[col] == 1)
    
    return {
        "messages": [
            {
                "role": "user",
                "content": user_msg
            }
        ],
        "reference_answer": ', '.join(labels)
    }

rft_data = [get_rft_example(example) for example in ds['validation']]
with open('emotion_rft_data.jsonl', 'w') as f:
    for item in rft_data:
        f.write(json.dumps(item) + '\n')

In [6]:
grader = {
  "name": "emotion_iou_grader",
  "type": "score_model",
  "input": [
    {
      "role": "user", 
      "content": "Calculate the Intersection over Union (IoU) score between the reference emotion list and model emotion list. IoU = intersection / union. Return just a floating point score between 0.0 and 1.0.\n\nReference emotions: {{item.reference_answer}}\n\nModel emotions: {{sample.output_text}}"
    }
  ],
  "model": "gpt-4o-2024-08-06",
  "sampling_params": {"seed": 42}
}

# Classification

In [7]:
system_msg = f"You are the emotion recognition system. Give a comma-separeted list of emotions, out of ('anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust') posibilities, that are present in the message:"
text = "I'm feeling overwhelmed and scared about tomorrow"

In [8]:
from openai import AzureOpenAI

endpoint = "https://murmur.openai.azure.com/"
# model_name = "gpt-4.1"
model_name = "o4-mini-emotions"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
)

response = client.beta.chat.completions.parse(
    model=model_name,
    messages=[
        {
            "role": "system",
            "content": system_msg,
        },
        {
            "role": "user",
            "content": text,
        }
    ],
    # response_format=EmotionFlags,
    # max_completion_tokens=800,
    # temperature=1.0,
    # top_p=1.0,
    # frequency_penalty=0.0,
    # presence_penalty=0.0,
)

response = response.choices[0].message
response.content

'fear, sadness'

In [9]:
import wandb
tags = ['emotions','SemEval2018']
wandb.init(project="emotions-llm",tags=set(tags))
wandb.config.update({"prompt":system_msg, "model": model_name, 'test_data':'SemEval2018'})

In [10]:
from tqdm import tqdm

texts = pd.Series(ds['test']['Tweet'])

responses = []
batch_size = 1
n_batch = (len(texts) + batch_size - 1) // batch_size

i = 0
with tqdm(total=len(texts),initial=i) as pbar:
    while i < n_batch:
        try:
            texts_batch = texts[i*batch_size:(i+1)*batch_size].reset_index().to_json(orient='records')
            response = client.beta.chat.completions.parse(
                model=model_name,
                # max_tokens = 4092,
                messages=[{"role":"system", "content":system_msg}, {"role": "user", "content": texts_batch}],
            )
            responses.append( (i,response.choices[0].message.content) )
        except Exception as e:
            print(f"\nError at batch {i}: {e}")
        
        i+= 1
        pbar.update(1)

  1%|          | 18/3259 [04:02<28:48:51, 32.01s/it]


Error at batch 17: Error code: 500 - {'error': {'message': 'The server had an error while processing your request. Sorry about that!', 'type': 'server_error', 'param': None, 'code': None}}


  1%|          | 25/3259 [05:24<10:25:39, 11.61s/it]


Error at batch 24: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': True, 'severity': 'medium'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


  1%|          | 30/3259 [07:08<27:58:42, 31.19s/it]

...

100%|█████████▉| 3244/3259 [2:28:25<00:41,  2.78s/it]


Error at batch 3243: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': True, 'severity': 'high'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


100%|█████████▉| 3248/3259 [2:28:36<00:29,  2.65s/it]


Error at batch 3247: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': True, 'severity': 'medium'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


100%|██████████| 3259/3259 [2:29:02<00:00,  2.74s/it]


In [11]:
y_pred_all = pd.DataFrame(responses,columns=['id','emotions'])
wandb.log({"all_predictions": wandb.Table(dataframe=y_pred_all)})
wandb.finish()

# Comparison

We see that the fine-tunning improves the IoU score from 0.45 to 0.56.

## Grund Truth

In [12]:
from sklearn.preprocessing import MultiLabelBinarizer

binarizer = MultiLabelBinarizer()
y_true = binarizer.fit_transform( ds['test']['labels'] )

# binarizer.classes_

## Models

In [13]:
import wandb
import numpy as np

entity = "mskorski-university-of-warsaw"
project = "emotions-llm"
tab_name = "all_predictions"

api = wandb.Api()

run_id = 'xb8y26xu'
run = api.run(f"{entity}/{project}/{run_id}")
print(run.config["model"])
artifact = api.artifact(f"{entity}/{project}/run-{run_id}-{tab_name}:latest")
table = artifact.get(tab_name)
table = pd.DataFrame(data=table.data,columns=table.columns)
y1 = table.set_index("id")
y1 = y1.reindex(index=np.arange(len(y_true)),fill_value='')

run_id = '4m9bpmfp'
run = api.run(f"{entity}/{project}/{run_id}")
print(run.config["model"])
artifact = api.artifact(f"{entity}/{project}/run-{run_id}-{tab_name}:latest")
table = artifact.get(tab_name)
table = pd.DataFrame(data=table.data,columns=table.columns)
y2 = table.set_index("id")
y2 = y2.reindex(index=np.arange(len(y_true)),fill_value='')

o4-mini-emotions


wandb:   1 of 1 files downloaded.  


gpt-4.1


wandb:   1 of 1 files downloaded.  


In [14]:
emotion_labels = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']

binarizer = MultiLabelBinarizer(classes=emotion_labels)
y1_t = binarizer.fit_transform( y1['emotions'].apply(lambda s: s.replace(' ','').split(',')) )
y2_t = binarizer.fit_transform( y2['emotions'].apply(lambda s: s.replace(' ','').split(',')) )

/home/skorsmac/research/apps/.apps/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['', '""', '("")', '(empty)', 'negativeemotion', 'sadnessemotion'] will be ignored
  warnings.warn(
/home/skorsmac/research/apps/.apps/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['', '"butthecontextsuggeststheopposite)', '"closestis\'joy\')', 'acceptance', 'admiration', 'alarm', 'amazement', 'amusement', 'amusement(impliedby"chuckles', 'annoyance', 'annoyed', 'anxiety', 'appreciation', 'assertiveness', 'awe', 'bitterness', 'boredom', 'but"anger"isindicatedbytheemojiandsarcastictone;"joy"mightseemimpliedby"exciting', 'calm', 'care', 'caution', 'compassion', 'concern', 'confidence', 'conflict', 'confusion', 'contentment', 'courage', 'curiosity', 'defensiveness', 'defiance', 'despair', 'determination(impliedwithinoptimism)', 'disappointment', 'disbelief', 'discomfort', 'distrust', 'empathy', 'excitement', 

In [15]:
def get_iou(y1,y2):
    return (y1*y2).sum(1) / (1e-6+np.maximum(y1,y2).sum(1))

print(get_iou(y1_t,y_true).mean(),get_iou(y2_t,y_true).mean())

0.5661630605003988 0.4494117255728142
